In [ ]:
import pandas as pd
import numpy as np
import torch

from src.forecaster import Forecaster
from src.models import LSTMHistoric, LSTMEncoderDecoder
from src.data import DataSource
from src.utils.scores_and_losses import MAE, RMSE, DILATE

In [ ]:
HISTORIC_COLS = ['prcp', 'tmax', 'tmin', 'vp', 'srad']
FUTURE_COLS   = ['prcp', 'tmax', 'tmin', 'vp', 'srad']
TARGET_COL    = 'target'
HORIZON       = 7
SEQ_LEN       = 30

train_source = DataSource(
    csv='data/data.csv',
    start='1980-01-01',
    end='2004-12-31',
    target_col=TARGET_COL,
    csv_index_col=1,
)
val_source = DataSource(
    csv='data/data.csv',
    start='2005-01-01',
    end='2009-12-31',
    target_col=TARGET_COL,
    csv_index_col=1,
)
test_source = DataSource(
    csv='data/data.csv',
    start='2010-01-01',
    end='2014-12-31',
    target_col=TARGET_COL,
    csv_index_col=1,
)

In [ ]:
from src.data.timeseries_dataset import TimeSeriesDataset
from src.data.normalization import compute_norm_stats

norm_stats = compute_norm_stats(
    sources=[train_source, val_source],
    seq_len=SEQ_LEN,
    horizon=HORIZON,
    historic_cols=HISTORIC_COLS,
    future_cols=FUTURE_COLS,
    patch_size=64,
)

ds = TimeSeriesDataset(
    sources=[train_source],
    seq_len=SEQ_LEN,
    horizon=HORIZON,
    historic_cols=HISTORIC_COLS,
    future_cols=FUTURE_COLS,
    patch_size=64,
    norm_stats=norm_stats,
)
sample = ds[0]
print(f"x_h: {sample['x_h'].shape}  x_f: {sample['x_f'].shape}  y: {sample['y'].shape}")
print(f'Dataset size: {len(ds)} windows')

In [ ]:
fc = Forecaster(
    model=LSTMHistoric,
    model_config={
        'LSTM_hidden_size': 16,
        'LSTM_num_layers': 1,
        'dropout_rate': 0.0,
    },
    name='LSTMHistoric_test',
    training_datasets=[train_source],
    validation_datasets=[val_source],
    historic_cols=HISTORIC_COLS,
    future_cols=FUTURE_COLS,
    forecasting_horizon=HORIZON,
    historic_input_sequence_length=SEQ_LEN,
    loss_function='mae',
    validation_score='rmse',
    minimize_validation_score=True,
    save_path='models',
    batch_size=1024,
    number_of_epochs=5,
    use_torch_compile=False,
    save_weights_every_n_epochs=1,
)

print(f'LSTMHistoric parameters: {fc.compute_number_of_parameters()}')

In [ ]:
fc.fit()

In [ ]:
fc2 = Forecaster(
    model=LSTMEncoderDecoder,
    model_config={
        'LSTM_hidden_size': 16,
        'LSTM_num_layers': 1,
        'dropout_rate': 0.0,
        'encoder_downscale_layer_size': 8,
    },
    name='LSTMEncoderDecoder_test',
    training_datasets=[train_source],
    validation_datasets=[val_source],
    historic_cols=HISTORIC_COLS,
    future_cols=FUTURE_COLS,
    forecasting_horizon=HORIZON,
    historic_input_sequence_length=SEQ_LEN,
    loss_function='mae',
    validation_score='rmse',
    minimize_validation_score=True,
    save_path='models',
    batch_size=1024,
    number_of_epochs=5,
    use_torch_compile=False,
    save_weights_every_n_epochs=1,
)

print(f'LSTMEncoderDecoder parameters: {fc2.compute_number_of_parameters()}')

In [ ]:
fc2.fit()